# 06. Preprocessing stats, Denmark-wide
## Project: Bicycle node network loop analysis

This notebook generates extra basic statistics from the network preprocessing steps for reporting purposes.  
Please select `denmark` as the `study_area` in the `config.yml`. 

Created: 2026-03-04  
Last modified: 2026-03-04

## Parameters

In [ ]:
%run -i setup_parameters.py
debug = True  # Set to True for extra plots and verbosity

## Functions

In [ ]:
%run -i functions.py

## Processing data

### Load data

#### 6 islands raw, before neatnetting

In [ ]:
nodes = gpd.GeoDataFrame()
edges = gpd.GeoDataFrame(columns=["objekt_id"])
for subarea in STUDY_AREA_COMBINED[STUDY_AREA]:
    e = gpd.read_file(PATH[subarea]["data_in_network_raw"] + "edges.gpkg")
    edges = pd.concat([edges, e])

edges.set_geometry(col="geometry", inplace=True)
if debug:
    edges.plot()

#### 6 islands after neatnetting, before preprocessing

In [ ]:
edges.items()

In [ ]:
nodes = gpd.GeoDataFrame()
edges = gpd.GeoDataFrame(columns=["objekt_id"])
Gs = []
for subarea in STUDY_AREA_COMBINED[STUDY_AREA]:
    e = gpd.read_file(PATH[subarea]["data_in_network"] + "edges_slope.gpkg")
    e["edge_id"] = e.index  # Make index the edge id
    edges = pd.concat([edges, e])
    n = gpd.read_file(PATH[subarea]["data_in_network"] + "nodes.gpkg")
    nodes = pd.concat([nodes, n])

    # Rename mm_len to weight for igraph
    e = e.rename(columns={"mm_len": "weight"})
    # Drop unused columns
    used_columns = {
        "node_start": (),
        "node_end": (),
        "weight": (),
        "edge_id": (),
        "max_slope": (),
        "geometry": (),
    }
    for c_name, _ in e.items():
        if c_name not in used_columns:
            del edges[c_name]
    # Reorder columns
    e = e[["node_start", "node_end", "weight", "edge_id", "max_slope", "geometry"]]
    e = e.dropna()  # Drop edges with None node_start or node_end
    G = ig.Graph.TupleList(
        e.itertuples(index=False),
        directed=False,
        weights=False,
        edge_attrs=["weight", "edge_id", "max_slope", "geometry"],
    )
    Gs.append(G)

edges.set_geometry(col="geometry", inplace=True)
if debug:
    edges.plot()
# Export data
edges.to_file(PATH["data_in_network"] + "edges.gpkg")

Nodes and links:

In [ ]:
len(nodes), len(edges)

Length in km:

In [ ]:
sum(edges.mm_len) / 1000

Components:

In [ ]:
components = []
for G in Gs:
    components.append(len(G.connected_components(mode="strong")))
print(components)
sum(components)

Blind ends:

In [ ]:
bes = []
for G in Gs:
    bes.append(len(G.vs.select(_degree_le=1)))
print(bes)
sum(bes)

Non-blind ends:

In [ ]:
nbes = []
for G in Gs:
    nbes.append(len(G.vs.select(_degree_ge=2)))
print(nbes)
sum(nbes)

#### 6 islands after preprocessing

In [ ]:
with lzma.open(PATH["data_out"] + "network_preprocessed.xz", "rb") as f:
    G = pickle.load(f)

Nodes and links:

In [ ]:
print(G.summary())

Length in km:

In [ ]:
sum(G.es["weight"]) / 1000

Components:

In [ ]:
len(G.connected_components(mode="strong"))

Blind ends:

In [ ]:
len(G.vs.select(_degree_le=1))

#### POIs

##### Raw POIs

In [ ]:
# Load data
pois = load_pois()

In [ ]:
pois.category.value_counts()

In [ ]:
len(pois)

##### Snapped POIs

In [ ]:
# Load data
with lzma.open(PATH["data_out"] + "pois_snapped.xz", "rb") as f:
    pois_snapped = pickle.load(f)

In [ ]:
pois_snapped.category.value_counts()

In [ ]:
len(pois_snapped)